# Démo : Vérification des réponses

**Contexte :** assistant du service client de Lagune Banque.

Une réponse fluide peut contenir un chiffre inventé ou une affirmation absente des documents. Cette démo compare trois contrôles sur quatre réponses dont les défauts sont connus :

1. la réponse cite-t-elle une source ? (une expression régulière)
2. les chiffres de la réponse figurent-ils dans les chunks ? (un calcul, sans modèle)
3. un modèle évaluateur vérifie-t-il que chaque affirmation est appuyée ?

**Sommaire**
- [Configuration](#configuration)
- [Chunks de référence](#chunks)
- [Réponses testées](#reponses)
- [Contrôles](#controles)
- [Résultats](#resultats)
- [Synthèse](#synthese)

```mermaid
flowchart LR
    A["Réponse générée"] --> N1["Niveau 1 : source citée"]
    N1 -->|passe| N2["Niveau 2 : chiffres dans les chunks"]
    N2 -->|passe| N3["Niveau 3 : juge"]
    N3 -->|passe| OK["Envoyée"]
    N1 -->|échec| B["Bloquée"]
    N2 -->|échec| B
    N3 -->|échec| B
```

Le but : voir ce que chaque niveau attrape, ce qu'il laisse passer, et ce qu'il coûte.

## <a id="configuration"></a>Configuration

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(next(p for p in [Path("code_partage"), Path("../code_partage"), Path("../../code_partage"), Path("../../../code_partage"), Path("../../starter/code_partage")] if (p / "rag.py").exists()).resolve()))
import rag
import rag_avance as ra
from config import prepare
from weaviate_store import connect
import re

In [ ]:
prepare(embedding=True, rerank=True)

## <a id="chunks"></a>Chunks de référence

Les chunks viennent de la recherche retenue au module : vectoriel puis reranker.

In [ ]:
documents = rag.load_corpus()
for d in documents:
    if d["fichier"].endswith(".pdf") and not d["avertissement"]:
        d["pages"] = rag.strip_headers(d["pages"])
index = rag.index_chunks(connect(), "demo3_verification", rag.split_documents(documents, 500, 75, "verif"))

QUESTION = "Quels sont les frais de retard du Prêt Express ?"
chunks = rag.rerank(QUESTION, rag.search(index, QUESTION, 20, rag.AUJOURDHUI), 3)
chunks_texte = rag.format_chunks(chunks)
print(chunks_texte[:700])

## <a id="reponses"></a>Réponses testées

On écrit les réponses à la main pour contrôler chaque défaut :

| Réponse | Défaut |
|---|---|
| A | aucun : chiffre exact, source citée |
| B | **chiffre inventé** (3 500 FCFA au lieu de 2 000), source citée |
| C | chiffre exact, mais **une affirmation absente des documents** (remboursement sous 30 jours), source citée |
| D | exacte, mais **sans aucune source** |

In [ ]:
REPONSES = {
    "A. Exacte": "Des frais de retard forfaitaires de 2 000 FCFA s'appliquent en cas d'impayé sur un Prêt Express [DOC-01].",
    "B. Chiffre inventé": "Des frais de retard forfaitaires de 3 500 FCFA s'appliquent en cas d'impayé sur un Prêt Express [DOC-01].",
    "C. Affirmation inventée": "Des frais de retard de 2 000 FCFA s'appliquent en cas d'impayé, remboursés sous 30 jours si vous contactez le service client [DOC-01].",
    "D. Sans source": "Des frais de retard forfaitaires de 2 000 FCFA s'appliquent en cas d'impayé sur un Prêt Express.",
}

## <a id="controles"></a>Contrôles

Le niveau 2 extrait les nombres de la réponse (un nombre peut contenir des espaces : « 2 000 ») et vérifie qu'ils figurent dans le texte des chunks. Le niveau 3 réutilise le juge de fidélité : le modèle, avec une sortie structurée (un schéma Pydantic), décompose la réponse en affirmations et indique pour chacune si les chunks l'appuient.

In [ ]:
from pydantic import BaseModel


def niveau_1(reponse):
    return bool(re.search(r"\[DOC-\d+\]", reponse))


def nombres(texte):
    return {re.sub(r"\s", "", n) for n in re.findall(r"\d[\d\s]*\d|\d", ra.normalize(texte))}


def niveau_2(reponse):
    contexte = nombres(" ".join(p["texte"] for p in chunks))
    return nombres(re.sub(r"\[DOC-\d+\]", "", reponse)) <= contexte


class Affirmation(BaseModel):
    texte: str
    appuyee: bool


class Faithfulness(BaseModel):
    affirmations: list[Affirmation]


PROMPT_JUGE = """Décompose la réponse en affirmations simples. Pour chacune, indique si les chunks l'appuient (appuyee vrai) ou non (faux).
Ce qui n'est pas écrit dans les chunks n'est pas appuyé, même si c'est plausible.

# Chunks
{chunks}

# Réponse
{reponse}"""


def niveau_3(reponse):
    result = ra.structured(PROMPT_JUGE.format(chunks=chunks_texte, reponse=reponse), Faithfulness)
    return bool(result and result.affirmations) and all(a.appuyee for a in result.affirmations)

In [ ]:
appels = ra.APPELS["llm"]
tableau = {nom: {"1. source citée": niveau_1(r), "2. chiffres dans les chunks": niveau_2(r), "3. juge": niveau_3(r)} for nom, r in REPONSES.items()}
import pandas as pd

pd.DataFrame(tableau).T.map(lambda ok: "passe" if ok else "bloquée")

## <a id="resultats"></a>Résultats

Chaque colonne est un niveau de vérification ; « bloquée » signifie que la réponse n'est pas envoyée. Les lignes B, C et D sont les réponses défectueuses : une vérification est utile si elle bloque les défauts qu'elle prétend attraper et laisse passer la réponse A.

In [ ]:
bloque = {niveau: [nom for nom, v in tableau.items() if not v[niveau]] for niveau in next(iter(tableau.values()))}
for niveau, noms in bloque.items():
    print(f"{niveau:32s} bloque : {', '.join(noms) or 'aucune'}")
print(f"Coût du niveau 3 : {ra.APPELS['llm'] - appels} appels au modèle pour {len(REPONSES)} réponses (les niveaux 1 et 2 n'en font aucun).")

## <a id="synthese"></a>Synthèse

- Le niveau 1 vérifie la **forme** : une source est citée, pas qu'elle dit vrai.
- Le niveau 2 attrape les chiffres inventés sans coûter un appel au modèle, mais ne voit pas une affirmation sans chiffre.
- Le niveau 3 attrape les affirmations inventées, au prix d'un appel supplémentaire par réponse.
- En production, on enchaîne : les niveaux peu coûteux filtrent d'abord, le juge ne regarde que ce qui reste.

Le TP3 utilise un autre contrôle, la citation d'extraits mot pour mot ; le TP4 mesure la fidélité sur un jeu complet de questions.